# 菜價預測（開發用 Notebook）

用來邊看資料/圖表邊調整預測邏輯，保留了 EDA 與單一產品的 ACF/PACF 視覺化，方便判斷挑 lag 的方式是否合理。

正式排程自動化執行的是 repo 內的 `price_prediction.py`，兩者核心邏輯完全一致（每個產品各自挑 lag、各自訓練、各自預測，不共用模型）；在這裡試出新邏輯後，記得同步更新 `price_prediction.py`。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.linear_model import LinearRegression

df = pd.read_csv('veg_prices_history.csv')
df['交易日期'] = pd.to_datetime(df['交易日期'], format='%Y/%m/%d', errors='coerce')
df.head()

### EDA：先看看資料長什麼樣子

In [ ]:
df.describe()

In [ ]:
tab_info = pd.DataFrame(df.dtypes).T.rename(index={0: 'column type'})
tab_info = pd.concat(
    [tab_info, pd.DataFrame(df.isnull().sum()).T.rename(index={0: 'null values'})],
    axis=0
)
tab_info

### 正規化產品名稱並依交易量重新加權合併

「產品名稱」欄位可能帶有品種/等級（例如「冬瓜 白皮」），先取第一個字詞當作基礎菜名，同一天同一種菜的資料再重新加權平均合併，避免同一種菜被拆成過多細碎、資料稀疏的分組，也讓菜名跟 `chat-_bot` 那邊比對用的簡化菜名一致。

In [ ]:
df['產品名稱'] = df['產品名稱'].str.strip().str.split().str[0]

df['加權合計'] = df['加權平均價(元/公斤)'] * df['總交易量(公斤)']
df = (
    df.groupby(['交易日期', '產品名稱'], as_index=False)
    .agg(加權合計=('加權合計', 'sum'), 總交易量公斤=('總交易量(公斤)', 'sum'))
)
df['加權平均價(元/公斤)'] = df['加權合計'] / df['總交易量公斤']
df = df[['交易日期', '產品名稱', '加權平均價(元/公斤)']]

df.set_index('交易日期', inplace=True)
df.index.name = '交易時間'
df.sort_index(inplace=True)

grouped_by_product = df.groupby('產品名稱')
print(f"共 {len(grouped_by_product)} 種蔬菜")

### 抽一種菜出來看價格走勢

可以把 `馬鈴薯` 換成任何想確認的菜名。

In [ ]:
sample_name = '馬鈴薯'
sample_df = df[df['產品名稱'] == sample_name].copy()

plt.figure(figsize=(10, 5))
plt.plot(sample_df.index, sample_df['加權平均價(元/公斤)'])
plt.xlabel('Trade Date')
plt.ylabel('Weighted Average Price (NTD/kg)')
plt.title(f'{sample_name} - Weighted Average Price Over Time')
plt.grid(True)
plt.tight_layout()
plt.show()

### 看單一產品的 ACF / PACF

用來直覺判斷該挑哪些落後期（lag）當特徵比較合理。正式流程裡每個產品會用 PACF 自動挑出最顯著的 lag，這裡先用圖確認一下判斷是否合理。

In [ ]:
ts_sample = pd.to_numeric(sample_df['加權平均價(元/公斤)'], errors='coerce').dropna()
nlags_sample = max(min(20, len(ts_sample) // 2 - 1), 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(ts_sample, lags=nlags_sample, ax=axes[0])
plot_pacf(ts_sample, lags=nlags_sample, ax=axes[1])
axes[0].set_title(f'{sample_name} - ACF')
axes[1].set_title(f'{sample_name} - PACF')
plt.tight_layout()
plt.show()

### 每個產品各自訓練 + 預測明日價格

跟 `price_prediction.py` 的正式邏輯一致：每個產品各自挑 lag、各自訓練線性迴歸模型、各自預測，不會共用別的產品的模型。

In [ ]:
MAX_TOP_LAGS = 3  # 每個產品最多選出幾個最顯著的 lag 當特徵
MIN_SAMPLES = MAX_TOP_LAGS + 2  # 資料筆數低於此門檻就跳過該產品

predictions = []

for name, group in grouped_by_product:
    ts = pd.to_numeric(group['加權平均價(元/公斤)'], errors='coerce').dropna()

    if len(ts) < MIN_SAMPLES:
        print(f"{name}: 資料太少（{len(ts)} 筆），跳過")
        continue

    nlags = min(20, len(ts) // 2 - 1)
    nlags = max(nlags, 1)
    pacf_vals = pacf(ts, nlags=nlags)
    top_lags = np.argsort(np.abs(pacf_vals[1:]))[::-1][:MAX_TOP_LAGS] + 1
    top_lags = sorted(top_lags)

    if len(ts) < max(top_lags) + 1:
        print(f"{name}: 資料不足以涵蓋所選 lag {top_lags}，跳過")
        continue

    X = pd.DataFrame({f'lag_{k}': ts.shift(k) for k in top_lags})
    valid = X.notna().all(axis=1)
    X_valid, y_valid = X[valid], ts[valid]

    lin_reg = LinearRegression()
    lin_reg.fit(X_valid, y_valid)
    print(f"{name}: 選用 lag {top_lags}，R² = {lin_reg.score(X_valid, y_valid):.2f}")

    last_lags = pd.DataFrame([ts.iloc[-np.array(top_lags)].values], columns=X.columns)
    y_pred = lin_reg.predict(last_lags)[0]

    predictions.append({
        "產品名稱": name,
        "預測明日菜價(元/公斤)": round(y_pred, 2)
    })

result_df = pd.DataFrame(predictions)
result_df.head(10)

### 存檔

確認上面結果都沒問題後，才寫回 CSV。⚠️ 這一格會直接覆蓋 repo 裡的 `veg_pred.csv`，測試階段建議先跳過或改存到別的檔名。

In [ ]:
result_df.to_csv("veg_pred.csv", index=False, encoding="utf-8-sig")
print(f"✅ 已完成，輸出 veg_pred.csv（共 {len(result_df)} 項產品）")